# 1. Setup

In [3]:
import logging
import re
import sys
from pathlib import Path
from typing import Any, Dict, List, Tuple

import pandas as pd


def _setup_project_path() -> Path:
    """Finds the project root folder and appends <root>/src to sys.path."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]

    for candidate in candidates:
        src_path = candidate / "src"
        package_path = src_path / "byeias"
        if package_path.exists():
            src_str = str(src_path)
            if src_str not in sys.path:
                sys.path.insert(0, src_str)
            return src_path

    raise RuntimeError("Could not find the project path. Expected a folder named 'src/byeias'.")

# Add the source path BEFORE importing custom modules
SRC_PATH = _setup_project_path()

# Custom package imports ('# noqa: E402' suppresses linter warnings for late imports)
from byeias.backend.classification.model_bias import BiasDetectionPipeline  # noqa: E402
from byeias.backend.config_loader import get_backend_config  # noqa: E402
from byeias.backend.extraction.text_extracter import PDFTextExtractor  # noqa: E402
from byeias.backend.llm_explanation.llm_communicator import LLMCommunicator  # noqa: E402

c:\Users\david\AppData\Local\pypoetry\Cache\virtualenvs\byeias-FF63A2ZF-py3.13\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

NameError: name 'logging' is not defined

# 2. Generate Testdata

In [ ]:
def build_test_samples(n: int = 5) -> List[Dict[str, str]]:
    """Generiert N Test-Beispiele für die Vorhersage."""
    samples = [
        {"context": "The topic in the classroom was mathematics.", "text": "Girls are naturally worse at math."},
        {"context": "A new team was formed in the company.", "text": "The collaboration was professional and respectful."},
        {"context": "In a debate about immigration.", "text": "All migrants are a burden."},
        {"context": "Many cultures were represented at the city festival.", "text": "The festival was open and inclusive."},
        {"context": "Roles were assigned in the project meeting.", "text": "Women should rather take on supporting tasks."},
    ]
    return samples[:n]


# 3. PDF Extraction

In [ ]:
def test_pdf_extraction(pdf_path_str: str) -> None:
    """Tests the PDFTextExtractor."""
    logger.info("--- Testing PDF Extraction ---")
    pdf_path = Path(pdf_path_str)
    
    if not pdf_path.exists():
        logger.error(f"File not found: {pdf_path.resolve()}")
        return

    extractor = PDFTextExtractor(language="german")
    sentences = extractor.extract_sentences(str(pdf_path))
    
    logger.info(f"Extracted sentences: {len(sentences)}")
    for i, sentence in enumerate(sentences[:5], start=1):
        print(f"{i}: {sentence}")

# 4. Classification (BERT)

In [ ]:
def test_model_inference(use_loaded_weights: bool = True) -> None:
    """Tests model prediction (inference), optionally with pre-loaded weights."""
    logger.info("--- Testing Model Inference ---")
    config = get_backend_config()
    pipeline = BiasDetectionPipeline(model_name=config.classification.model_name)

    if use_loaded_weights:
        weights_path = Path(config.classification.best_model_path)
        if weights_path.exists():
            logger.info(f"Loading existing weights from: {weights_path}")
            try:
                pipeline.load_model(str(weights_path))
                logger.info("Model loaded successfully!")
            except Exception as e:
                logger.error(f"Critical error loading weights: {e}", exc_info=True)
                return
        else:
            logger.warning(f"Weights file not found at {weights_path}. Using base model instead.")

    test_samples = build_test_samples()
    test_contexts = [s["context"] for s in test_samples]
    test_texts = [s["text"] for s in test_samples]

    logger.info("Starting predictions...")
    predictions = pipeline.predict(context_texts=test_contexts, target_texts=test_texts)

    print("\n--- INFERENCE RESULTS ---")
    for idx, res in enumerate(predictions, start=1):
        sexism_flag = "YES" if res['sexism_prediction'] == 1 else "NO"
        racism_flag = "YES" if res['racism_prediction'] == 1 else "NO"
        
        print(f"Sample {idx}:")
        print(f"  Context: '{res['context']}'")
        print(f"  Text:    '{res['text']}'")
        print(f"  -> Sexism: {sexism_flag}")
        print(f"  -> Racism: {racism_flag}\n")

# 5. LLM Communication (Mistral)

In [ ]:
def test_llm_communication() -> None:
    """Tests communication with the LLM for generating bias explanations."""
    logger.info("--- Testing LLM Communication ---")
    context_before = "Im Klassenzimmer ging es um Mathematik."
    flagged_sentence = "Mädchen sind von Natur aus schlechter in Mathe."
    context_after = "Die Lehrerin erklärte die nächste Aufgabe."

    llm = LLMCommunicator()
    try:
        result = llm.explain_bias(context_before, flagged_sentence, context_after)
        print("\nLLM Result:")
        print(result)
    except Exception as e:
        logger.error(f"Error during LLM communication: {e}", exc_info=True)

# 6. Full Pipeline

In [ ]:
def test_full_pipeline(input_text: str) -> List[Dict[str, Any]]:
    """Tests the complete processing pipeline: Sentence splitting -> Inference -> LLM Explanation."""
    logger.info("--- Testing Complete Pipeline ---")
    
    # Split text into sentences (simple logic based on punctuation)
    raw_sentences = re.split(r'(?<=[.!?])\s+', input_text.strip())
    sentences = [s.strip() for s in raw_sentences if s.strip()]

    logger.info(f"Input text split into {len(sentences)} sentences.")
    if not sentences:
        logger.warning("No sentences found to process.")
        return []

    # Initialize components
    config = get_backend_config()
    pipeline = BiasDetectionPipeline(model_name=config.classification.model_name)
    
    # Optional: Try to load pre-trained weights
    try:
        pipeline.load_model(config.classification.best_model_path)
    except Exception as e:
        logger.warning(f"Could not load weights, continuing with base model. Reason: {e}")

    llm = LLMCommunicator()

    # Prepare context lists
    context_texts, target_texts = [], []
    contexts_before, contexts_after = [], []

    for i in range(len(sentences)):
        target = sentences[i]
        before = sentences[i-1] if i > 0 else ""
        after = sentences[i+1] if i + 1 < len(sentences) else ""
        
        combined_context = f"{before} {after}".strip()
        
        target_texts.append(target)
        context_texts.append(combined_context)
        contexts_before.append(before)
        contexts_after.append(after)

    # Generate predictions
    logger.info("Starting classification for all sentences...")
    inference_results = pipeline.predict(context_texts=context_texts, target_texts=target_texts)

    explanations = []
    
    # Evaluate results and fetch LLM explanations for flagged sentences
    for i, result in enumerate(inference_results):
        is_sexist = result["sexism_prediction"] == 1
        is_racist = result["racism_prediction"] == 1
        
        if is_sexist or is_racist:
            logger.info(f"[BIAS FOUND] in sentence {i+1}: '{result['text']}'")
            bias_type = "Sexism" if is_sexist else "Racism"
            
            try:
                explanation = llm.explain_bias(
                    context_before=contexts_before[i],
                    flagged_sentence=result["text"],
                    context_after=contexts_after[i],
                )
            except Exception as e:
                logger.error(f"LLM Error at sentence {i+1}: {e}")
                explanation = "Explanation could not be generated."

            explanations.append({
                "sentence_index": i + 1,
                "flagged_sentence": result["text"],
                "bias_type": bias_type,
                "llm_explanation": explanation
            })

    return explanations

# 7. Tests

In [ ]:

# 1. PDF Extraction
# test_pdf_extraction("../../../../data/....pdf")
    
# # 3. Model Inference (with or without saved weights)
# test_model_inference(use_loaded_weights=False) 
    
# # 4. LLM Communication
# test_llm_communication()
    
# 5. Full Pipeline
sample_text = (
    "The Topic is about payment. Girls get paid less. "
    "The teacher explained the next task. Everyone was listening."
)
results = test_full_pipeline(sample_text)
if results:
    print("\n" + "="*50)
    print("FINAL SUMMARY")
    print("="*50)
    for res in results:
        print(f"Sentence {res['sentence_index']} ({res['bias_type']}): {res['flagged_sentence']}")
        print(f"Explanation:\n{res['llm_explanation']}\n")
        print("-" * 50)
else:
    print("\nNo bias found in the text or processing failed.")